# Credit Agreement Analyzer - Phase 4 Validation
Testing Q&A engine against the real Ribbon Communications credit agreement.

Prerequisites: Ollama running locally with `llama3:8b` loaded.

In [1]:
import contextlib
import time
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker, build_search_text
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))

c:\Users\kbott\Projects\credit-analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Rebuild Pipeline (Phase 1-3)
Re-run extraction through retrieval setup.

In [2]:
# Phase 1: Extract, detect, parse, chunk
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)


defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
preamble_sections = [s for s in sections if s.section_type == "preamble"]
preamble_text = preamble_sections[0].text if preamble_sections else None
print(f"Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

Pages: 289 | Sections: 157 | Terms: 391 | Chunks: 705


In [3]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()}")
print(f"Available: {llm.is_available()}")

Provider: claude-sonnet-4-20250514
Available: True


In [4]:
# Phase 3: Retrieval layer
embedder = Embedder()

start = time.time()
embeddings = embedder.embed([build_search_text(c) for c in chunks])
print(f"Embedded {len(chunks)} chunks in {time.time() - start:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_p4")
store.create_collection("ribbon_p4")
store.add_chunks("ribbon_p4", chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)
print("Retrieval layer ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1296.27it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 705 chunks in 24.1s
Retrieval layer ready


## Step 1: Basic Q&A
Test single questions with no conversation history.

In [5]:
qa = QAEngine(retriever=retriever, llm=llm)
if preamble_text:
    qa.set_preamble(preamble_text)
print("QAEngine initialized")

QAEngine initialized


In [6]:
def print_response(resp: QAResponse) -> None:
    """Display a QAResponse in a readable format."""
    print(f"Answer:\n{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  Section {s.section_id} - {s.section_title} (pp. {pages})")
        if s.relevant_text_snippet:
            print(f"    Snippet: {s.relevant_text_snippet[:120]}...")
    print(f"\nChunks retrieved: {len(resp.retrieved_chunks)}")
    print(f"LLM time: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used}")
    print("-" * 80)

In [7]:
# Q1: Total revolving commitment (exact dollar amount)
qa.clear_history()
print("Q: What is the total revolving commitment amount?\n")
resp = qa.ask("What is the total revolving commitment amount?", "ribbon_p4")
print_response(resp)

Q: What is the total revolving commitment amount?

Answer:
The total revolving commitment amount is $35,000,000.

This is stated in multiple places in the credit agreement:

1. In the recitals, which specify that the Lenders have agreed to extend credit facilities "consisting of a term loan facility in the aggregate principal amount of $350,000,000, and a revolving loan facility to the Borrower in an aggregate principal amount of $35,000,000, including a letter of credit sub-facility"

2. In the definition of "Total Revolving Commitments" (Section 1.1), which states "The original amount of the Total Revolving Commitments is $35,000,000"

3. In the definition of "Revolving Commitment" (Section 1.1), which also confirms "The original amount of the Total Revolving Commitments is $35,000,000"

The revolving facility includes a letter of credit sub-facility, with the L/C Commitment being a sublimit of the Total Revolving Commitments rather than additional capacity.

Confidence: HIGH
Sources

In [8]:
# Q2: Financinal covenants (list them)
qa.clear_history()
print("Q: What financial covenants are included in the agreement?\n")
resp = qa.ask("What financial covenants are included in the agreement?", "ribbon_p4")
print_response(resp)

Q: What financial covenants are included in the agreement?

Answer:
Based on the provided context, the credit agreement includes one financial covenant:

Maximum Consolidated Net Leverage Ratio (Section 7.1(a))

The borrower must maintain a Consolidated Net Leverage Ratio that does not exceed the following thresholds, tested quarterly starting September 30, 2024:

1. September 30, 2024 through December 31, 2025: 4.75:1.00
2. March 31, 2026 and thereafter: 4.00:1.00

The covenant steps down from 4.75x to 4.00x starting in Q1 2026, providing some initial flexibility before tightening.

Equity Cure Provision (Section 7.1(b))

The agreement includes an equity cure mechanism that allows the borrower to inject cash equity contributions to remedy covenant breaches. Key features include:

- Must be contributed within 10 business days after financial statements are due
- The equity contribution gets added to Consolidated Adjusted EBITDA for covenant calculation purposes
- During the 10-day cure

In [9]:
# Q3: Incremental debt capacity (your core use case)
qa.clear_history()
print("Q: How much incremental debt can the borrower incur?\n")
resp = qa.ask("How much incremental debt can the borrower incur?", "ribbon_p4")
print_response(resp)

Q: How much incremental debt can the borrower incur?

Answer:
The borrower can incur incremental debt up to the "Available Incremental Amount," which is defined as the sum of two components:

1. Fixed Incremental Amount
- The greater of $50,000,000 or 50% of Consolidated Adjusted EBITDA from the most recent quarter
- Reduced by any previously incurred Incremental Facilities under this basket (Section 1.1)

2. Ratio Incremental Amount 
- Additional debt capacity based on leverage ratio tests, with different thresholds depending on security structure:
  - Pari passu secured debt: Up to 2.90x Consolidated First Lien Net Leverage Ratio
  - Junior lien secured debt: Up to 3.40x Consolidated Secured Net Leverage Ratio  
  - Unsecured debt: Up to 3.90x Consolidated Net Leverage Ratio
- Calculated on a pro forma basis assuming full utilization (Section 1.1)

Key Constraints:
- For revolving commitment increases, the ratio of total revolving commitments to term loans cannot exceed 10% (Section 

In [10]:
# Q4: Restricted payments (complex multi-basket answer)
qa.clear_history()
print("Q: What restricted payments are permitted under the credit agreement?\n")
resp = qa.ask("What restricted payments are permitted under the credit agreement?", "ribbon_p4")
print_response(resp)

Q: What restricted payments are permitted under the credit agreement?

Answer:
Based on the credit agreement, the following restricted payments are permitted under Section 7.6, provided no Event of Default has occurred and is continuing or would result from the payment:

SUBSIDIARY-TO-PARENT DISTRIBUTIONS
1. Any Subsidiary may make restricted payments to any Loan Party
2. Non-Loan Party subsidiaries may make restricted payments to other Group Members or to any holder of their capital stock on a pro rata basis (or more favorably to Loan Parties)

EMPLOYEE-RELATED PAYMENTS
3. Purchase of capital stock/options from current/former directors, officers, or employees with annual limits:
   - $2.5 million per fiscal year (unused amounts carry forward)
   - Additional $5 million if Consolidated Net Leverage Ratio ≤ 2.40:1.00 after the payment
   - Additional $10 million if Consolidated Net Leverage Ratio ≤ 1.90:1.00 after the payment

4. Stock dividends payable solely in common stock
5. Employe

In [11]:
# Q5: NOT FOUND test - question about something unlikely to be in the doc
qa.clear_history()
print("Q: What are the details of the borrower's pet insurance policy?\n")
resp = qa.ask("What are the details of the borrower's pet insurance policy?", "ribbon_p4")
print_response(resp)

Q: What are the details of the borrower's pet insurance policy?

Answer:
Based on my review of the provided credit agreement context, there is no information about the borrower's pet insurance policy. The agreement does not contain any provisions, requirements, or details regarding pet insurance coverage.

The insurance-related provisions in the agreement focus on standard commercial insurance requirements:

1. Section 4.24 requires that all insurance maintained by the Loan Parties be in full force and effect with premiums paid, and that each Loan Party maintains insurance with financially sound and reputable companies on its property in amounts and against risks usually insured against by similar businesses.

2. Section 6.6 requires maintenance of insurance on property in amounts and against risks (including public liability, product liability and business interruption) as are usually insured against by companies in the same or similar business.

The agreement does not address pet ins

## Step 2: Conversation Follow-ups
Test that question reformulation works for multi-turn Q&A.

In [12]:
qa.clear_history()

# Turn 1: Establish context
print("Turn 1: What is the term loan facility size?\n")
resp1 = qa.ask("What is the term loan facility size?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up using "it" (requires reformulation)
print("\nTurn 2: What is the interest rate on it?\n")
resp2 = qa.ask("What is the interest rate on it?", "ribbon_p4")
print_response(resp2)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What is the term loan facility size?

Answer:
The term loan facility size is $350,000,000.

This is stated in the Recitals section, which indicates that "the Lenders have agreed to extend certain credit facilities to the Borrower...in an aggregate principal amount not to exceed $385,000,000, consisting of a term loan facility in the aggregate principal amount of $350,000,000, and a revolving loan facility to the Borrower in an aggregate principal amount of $35,000,000."

This is further confirmed in the definition of "Term Commitment" which states: "The original aggregate principal amount of the Term Commitments on the Closing Date is $350,000,000."

Confidence: HIGH
Sources (1):
  Section 1.1 - Defined Terms (pp. 54)
    Snippet: “Term Facility”: the Term Commitments and the Term Loans made thereunder.  “Term Lender”: each Lender that has a Term Co...

Chunks retrieved: 15
LLM time: 4.9s | Tokens: 184
----------------------------------------------------------------------------

In [13]:
qa.clear_history()

# Turn 1: Ask about covenants broadly
print("Turn 1: What financial covenants does this agreement have?\n")
resp1 = qa.ask("What financial covenants does this agreement have?", "ribbon_p4")
print_response(resp1)

# Turn 2: Follow-up referencing the previous answer
print("\nTurn 2: Are there any step-downs in those levels?\n")
resp2 = qa.ask("Are there any step-downs in those levels?", "ribbon_p4")
print_response(resp2)

# Turn 3: Another follow-up
print("\nTurn 3: What happens if the borrower breaches them?\n")
resp3 = qa.ask("What happens if the borrower breaches them?", "ribbon_p4")
print_response(resp3)

print(f"Conversation history length: {qa.history_length}")

Turn 1: What financial covenants does this agreement have?

Answer:
Based on the provided context, this credit agreement contains one financial covenant:

Financial Maintenance Covenant

The agreement has a financial condition covenant under Section 7.1(a), though the specific covenant terms are not detailed in the provided context. However, I can see that:

1. The covenant is tested based on "the last day of any period of four consecutive trailing fiscal quarters of Holdings" (Section 7.1)

2. There is an equity cure mechanism available if the borrower fails to comply with the financial covenant (Section 7.1(b))

Equity Cure Provisions

If the borrower fails the financial covenant test, they can make a cash common equity contribution within 10 business days after financial statements are due. This "Specified Equity Contribution" gets included in the calculation of Consolidated Adjusted EBITDA solely for covenant compliance purposes.

Key restrictions on the equity cure:
- No loans can

## Step 3: Context Assembly Inspection
Look at the raw prompt being sent to the LLM to verify structure.

In [14]:
from credit_analyzer.generation.prompts import build_context_prompt

# Manually retrieve and build context to inspect
result = retriever.retrieve(
    query="What is the total revolving commitment amount?",
    document_id="ribbon_p4",
    top_k=5,
)

prompt = build_context_prompt(
    chunks=result.chunks,
    definitions=result.injected_definitions,
    history=[],
    question="What is the total revolving commitment amount?",
)

print(f"Prompt length: {len(prompt)} chars")
print(f"Approximate tokens: ~{len(prompt) // 4}")
print("\n" + "=" * 80)
print(prompt)
print("=" * 80)

Prompt length: 6755 chars
Approximate tokens: ~1688

=== CONTEXT FROM CREDIT AGREEMENT ===

--- Source: Defined Terms (Section 1.1, Pages 55) ---
“Total Revolving Commitments”: at any time, the aggregate amount of the Revolving Commitments then in effect. The
original amount of the Total Revolving Commitments is $35,000,000. The L/C Commitment is a sublimit of the Total Revolving
Commitments.

--- Source: Defined Terms (Section 1.1, Pages 50) ---
“Revolving Commitment”: as to any Lender, the obligation of such Lender, if any, to make Revolving Loans in an aggregate
principal amount not to exceed the amount set forth under the heading “Revolving Commitment” opposite such Lender’s name on
Schedule 1.1A or in the Assignment and Assumption, Incremental Joinder, Extension Amendment or Refinancing Amendment, as
applicable, pursuant to which such Lender became a party hereto, as the same may be changed from time to time pursuant to the terms
hereof (including in connection with assignments an

## Step 4: Timing & Token Budget
Verify we're within the 8K context window and check response times.

In [15]:
test_questions = [
    "What is the total revolving commitment amount?",
    "What are the financial covenant test levels?",
    "How much incremental debt can the borrower incur?",
    "Who is the administrative agent?",
    "What are the mandatory prepayment triggers?",
]

print(f"{'Question':<55} {'Time':>6} {'Tokens':>7} {'Conf':>6}")
print("-" * 80)

for q in test_questions:
    qa.clear_history()
    start = time.time()
    resp = qa.ask(q, "ribbon_p4")
    elapsed = time.time() - start
    print(
        f"{q:<55} {elapsed:>5.1f}s {resp.llm_response.tokens_used:>7} {resp.confidence:>6}"
    )

Question                                                  Time  Tokens   Conf
--------------------------------------------------------------------------------
What is the total revolving commitment amount?           23.8s     288   HIGH
What are the financial covenant test levels?             20.2s     422   HIGH
How much incremental debt can the borrower incur?        10.6s     442   HIGH
Who is the administrative agent?                         28.6s     274   HIGH
What are the mandatory prepayment triggers?              27.6s     684   HIGH


## Cleanup

In [16]:
store.delete_collection("ribbon_p4")
print("Cleaned up test collection")

Cleaned up test collection
